# 🥗 The "Sugar Trap" — Market Gap Analysis
**AmaliTech Data Engineering Challenge — Project 2**

**Client:** Helix CPG Partners  
**Dataset:** [Open Food Facts](https://world.openfoodfacts.org/data)

---
### Notebook Structure
- **Section 0:** Setup & Data Loading
- **Story 1:** Data Ingestion & Cleaning
- **Story 2:** Category Wrangler — High-Level Buckets
- **Story 3:** Nutrient Matrix Visualization
- **Story 4:** Market Gap Recommendation
- **Bonus:** Hidden Gem — Protein Source Ingredients
- **Candidate's Choice:** Nutrition Score vs Market Position
---

## Section 0: Setup & Data Loading

In [40]:
!pip install plotly wordcloud --quiet
print('✅ Libraries ready')

✅ Libraries ready


In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Imports done')

✅ Imports done


In [42]:
# ─────────────────────────────────────────────────────────────
# Download Open Food Facts dataset
# Full file is 3GB+ — we load only the first 500,000 rows
# ─────────────────────────────────────────────────────────────
import os

DATA_FILE = 'en.openfoodfacts.org.products.csv'

# Check if already downloaded
if not os.path.exists(DATA_FILE):
    print('Downloading Open Food Facts dataset...')
    print('This may take a few minutes (~500MB download).')
    import urllib.request
    url = 'https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz'
    urllib.request.urlretrieve(url, 'openfoodfacts.csv.gz')
    print('✅ Downloaded. Now loading first 500k rows...')
else:
    print(f'✅ Found {DATA_FILE} locally')

This may take a few minutes (~500MB download).
✅ Downloaded. Now loading first 500k rows...


In [43]:
# ─────────────────────────────────────────────────────────────
# Load only needed columns from the first 500,000 rows
# This avoids memory issues with the 3GB+ full dataset
# ─────────────────────────────────────────────────────────────
COLS_NEEDED = [
    'product_name',
    'categories_tags',
    'ingredients_text',
    'sugars_100g',
    'proteins_100g',
    'fat_100g',
    'fiber_100g',
    'energy_100g',
    'nutrition_grade_fr',
    'nutriscore_score',
    'countries_tags',
    'brands'
]

# Try .gz first (downloaded compressed), fall back to .csv
try:
    raw = pd.read_csv(
        'openfoodfacts.csv.gz',
        sep='\t',
        usecols=lambda c: c in COLS_NEEDED,
        nrows=500_000,
        low_memory=False,
        on_bad_lines='skip'
    )
    print('Loaded from .gz')
except Exception:
    raw = pd.read_csv(
        DATA_FILE,
        sep='\t',
        usecols=lambda c: c in COLS_NEEDED,
        nrows=500_000,
        low_memory=False,
        on_bad_lines='skip'
    )
    print('Loaded from .csv')

print(f'Shape: {raw.shape}')
display(raw.head(3))

Loaded from .gz
Shape: (500000, 11)


,product_name,brands,categories_tags,countries_tags,ingredients_text,nutriscore_score,energy_100g,fat_100g,sugars_100g,fiber_100g,proteins_100g
0,Limonade artisanale a la rose,NaN,NaN,en:france,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M&amp;M white,Fitpiggy,NaN,en:france,"Weizenmehl, Rapsöl, Speisesalz, 1,7% Meersalz,...",NaN,NaN,NaN,NaN,NaN,NaN
2,Chocolate n3,Jeff de Bruges,NaN,en:france,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---
## Story 1: Data Ingestion & "The Clean Up" 🧹
**Goal:** Remove products with erroneous or missing nutritional data.

**Cleaning Strategy:**
1. Drop rows missing `product_name`, `sugars_100g`, or `proteins_100g`
2. Remove biologically impossible values (values >100g per 100g)
3. Remove extreme outliers (>3 standard deviations)
4. Keep only snack-relevant products (filter by categories_tags)

In [44]:
# ─────────────────────────────────────────────────────────────
# Step 1a: Missing value analysis
# ─────────────────────────────────────────────────────────────
print('=== Missing Value Summary ===')
missing = raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(raw) * 100).round(1)
print(pd.DataFrame({'Missing': missing, 'Pct': missing_pct}))

print(f'\nRaw rows: {len(raw):,}')

=== Missing Value Summary ===
                  Missing   Pct
fiber_100g         422435 84.50
sugars_100g        394575 78.90
fat_100g           391484 78.30
proteins_100g      391343 78.30
energy_100g        390509 78.10
nutriscore_score   265498 53.10
ingredients_text   232257 46.50
categories_tags    231436 46.30
brands             180575 36.10
product_name        15693  3.10
countries_tags       1735  0.30

Raw rows: 500,000


In [45]:
# ─────────────────────────────────────────────────────────────
# Step 1b: Drop essential nulls
# ─────────────────────────────────────────────────────────────
df = raw.copy()

# Drop rows with missing required fields
df = df.dropna(subset=['product_name', 'sugars_100g', 'proteins_100g'])
print(f'After dropping missing essentials: {len(df):,}')

# Drop rows with empty product_name
df = df[df['product_name'].str.strip().str.len() > 0]
print(f'After removing blank product names: {len(df):,}')

# Ensure numeric types
numeric_cols = ['sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g', 'energy_100g']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

After dropping missing essentials: 103,670
After removing blank product names: 103,670


In [46]:
# ─────────────────────────────────────────────────────────────
# Step 1c: Remove biologically impossible values
# Per 100g: sugars + proteins + fat + fiber cannot exceed 100g
# ─────────────────────────────────────────────────────────────

# Hard limits: no nutrient can be > 100g per 100g of product
df = df[
    (df['sugars_100g'].between(0, 100)) &
    (df['proteins_100g'].between(0, 100)) &
    (df['fat_100g'].fillna(0).between(0, 100))
]
print(f'After impossible value filter: {len(df):,}')

# Soft outlier filter: remove extreme z-scores (>4 std) for sugar & protein
for col in ['sugars_100g', 'proteins_100g']:
    mean, std = df[col].mean(), df[col].std()
    df = df[np.abs(df[col] - mean) <= 4 * std]

print(f'After z-score outlier removal:   {len(df):,}')
print(f'\nRows removed total: {len(raw) - len(df):,} ({(1 - len(df)/len(raw))*100:.1f}%)')

print('\n=== Cleaned Data Stats ===')
display(df[['sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g']].describe().round(2))

After impossible value filter: 103,497
After z-score outlier removal:   101,912

Rows removed total: 398,088 (79.6%)

=== Cleaned Data Stats ===


,sugars_100g,proteins_100g,fat_100g,fiber_100g
count,101912.00,101912.00,101713.00,74295.00
mean,12.56,7.64,11.54,2.97
std,17.65,7.51,15.40,5.20
min,0.00,0.00,0.00,0.00
25%,1.05,2.00,0.70,0.00
50%,4.65,6.00,4.96,1.64
75%,15.79,10.53,17.70,3.60
max,86.70,47.80,100.00,300.00


---
## Story 2: The Category Wrangler 📂
**Goal:** Parse messy `categories_tags` and assign readable high-level categories.

**5 Primary Categories:**
1. Chocolate & Candy
2. Chips & Savoury Snacks
3. Protein & Health Bars
4. Biscuits & Cookies
5. Cereals & Granola
6. Dairy & Yogurt
7. Other Snacks

In [47]:
# ─────────────────────────────────────────────────────────────
# Step 2a: Parse categories_tags
# Example: "en:snacks,en:sweet-snacks,en:chocolate" → list
# ─────────────────────────────────────────────────────────────
def clean_tag(tag):
    """Remove language prefix and clean tag"""
    tag = str(tag).lower().strip()
    tag = re.sub(r'^[a-z]{2}:', '', tag)  # remove en: fr: etc.
    tag = tag.replace('-', ' ').replace('_', ' ')
    return tag

def parse_categories(cat_string):
    """Parse comma-separated categories string into a list"""
    if pd.isna(cat_string) or str(cat_string).strip() == '':
        return []
    return [clean_tag(t) for t in str(cat_string).split(',')]

df['categories_list'] = df['categories_tags'].apply(parse_categories)
df['categories_str']  = df['categories_list'].apply(lambda x: ' '.join(x))

# Show most common tags
all_tags = [tag for tags in df['categories_list'] for tag in tags]
print('Top 20 most common category tags:')
display(pd.Series(Counter(all_tags)).nlargest(20))

Top 20 most common category tags:


,0
plant based foods and beverages,20911
plant based foods,19433
cereals and potatoes,12275
snacks,8416
breads,7137
dairies,6615
fermented foods,5833
fermented milk products,5784
sweet snacks,5645
desserts,5285


In [48]:
# ─────────────────────────────────────────────────────────────
# Step 2b: Assign Primary Category via keyword matching
# Priority order matters — first match wins
# ─────────────────────────────────────────────────────────────
CATEGORY_RULES = [
    ('Protein & Health Bars', ['protein bar', 'protein bar', 'energy bar', 'health bar',
                                'nut bar', 'cereal bar', 'sport', 'fitness']),
    ('Chocolate & Candy',     ['chocolate', 'candy', 'confectionery', 'caramel',
                                'gummy', 'marshmallow', 'toffee', 'fudge', 'truffle']),
    ('Chips & Savoury Snacks',['chips', 'crisps', 'popcorn', 'pretzel', 'cracker',
                                'savoury snack', 'savory snack', 'pork rind', 'tortilla']),
    ('Biscuits & Cookies',    ['biscuit', 'cookie', 'wafer', 'shortbread', 'digestive']),
    ('Cereals & Granola',     ['cereal', 'granola', 'muesli', 'oat', 'porridge', 'breakfast']),
    ('Dairy & Yogurt',        ['yogurt', 'yoghurt', 'dairy', 'cheese snack', 'milk snack']),
    ('Nuts & Seeds',          ['nut', 'almond', 'cashew', 'peanut', 'seed', 'trail mix',
                                'pistachio', 'walnut', 'sunflower']),
    ('Dried Fruit & Fruit Snacks', ['dried fruit', 'fruit snack', 'raisin', 'apricot',
                                     'fig', 'date', 'fruit bar', 'fruit leather']),
]

def assign_category(cat_str):
    for category, keywords in CATEGORY_RULES:
        if any(kw in cat_str for kw in keywords):
            return category
    return 'Other Snacks'

df['primary_category'] = df['categories_str'].apply(assign_category)

print('=== Primary Category Distribution ===')
cat_dist = df['primary_category'].value_counts()
display(cat_dist)

# Keep snack-relevant products (drop if 'Other Snacks' and no snack tag)
snack_keywords = ['snack', 'sweet', 'treat', 'dessert', 'confection']
is_snack = df['categories_str'].apply(lambda x: any(k in x for k in snack_keywords))
df = df[is_snack | (df['primary_category'] != 'Other Snacks')].copy()

print(f'\nFinal dataset after snack filter: {len(df):,} products')

=== Primary Category Distribution ===


,count
primary_category,
Other Snacks,76172
Cereals & Granola,12211
Dairy & Yogurt,4743
Chips & Savoury Snacks,2391
Chocolate & Candy,2092
Nuts & Seeds,1980
Biscuits & Cookies,1525
Protein & Health Bars,466
Dried Fruit & Fruit Snacks,332



Final dataset after snack filter: 30,189 products


In [49]:
# ─────────────────────────────────────────────────────────────
# Story 2 Chart: Category distribution
# ─────────────────────────────────────────────────────────────
cat_counts = df['primary_category'].value_counts().reset_index()
cat_counts.columns = ['Category', 'Count']

fig = px.bar(
    cat_counts, x='Count', y='Category',
    orientation='h',
    color='Count',
    color_continuous_scale='Viridis',
    title='🍫 Product Count by Category',
    text='Count'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=450, coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
fig.show()
fig.write_html('story2_categories.html')
print('✅ Chart saved')

✅ Chart saved


---
## Story 3: The Nutrient Matrix Visualization 📊
**Goal:** Scatter plot of Sugar vs Protein to identify the "Empty Quadrant" (High Protein + Low Sugar).

In [50]:
# ─────────────────────────────────────────────────────────────
# Step 3a: Compute quadrant thresholds (median split)
# ─────────────────────────────────────────────────────────────
sugar_median   = df['sugars_100g'].median()
protein_median = df['proteins_100g'].median()

print(f'Sugar median:   {sugar_median:.1f}g per 100g')
print(f'Protein median: {protein_median:.1f}g per 100g')

def get_quadrant(row):
    high_sugar   = row['sugars_100g']   >= sugar_median
    high_protein = row['proteins_100g'] >= protein_median
    if high_protein and not high_sugar:
        return '🌟 High Protein / Low Sugar (TARGET)'
    elif high_protein and high_sugar:
        return 'High Protein / High Sugar'
    elif not high_protein and high_sugar:
        return 'Low Protein / High Sugar (CROWDED)'
    else:
        return 'Low Protein / Low Sugar'

df['quadrant'] = df.apply(get_quadrant, axis=1)
print('\nQuadrant distribution:')
print(df['quadrant'].value_counts())

Sugar median:   7.7g per 100g
Protein median: 7.0g per 100g

Quadrant distribution:
quadrant
🌟 High Protein / Low Sugar (TARGET)    9584
Low Protein / High Sugar (CROWDED)     9545
High Protein / High Sugar              5570
Low Protein / Low Sugar                5490
Name: count, dtype: int64


In [51]:
# ─────────────────────────────────────────────────────────────
# Story 3 Main Chart: Sugar vs Protein Scatter
# ─────────────────────────────────────────────────────────────
sample = df.sample(min(8000, len(df)), random_state=42)

quadrant_colors = {
    '🌟 High Protein / Low Sugar (TARGET)': '#2ecc71',
    'High Protein / High Sugar':            '#3498db',
    'Low Protein / High Sugar (CROWDED)':   '#e74c3c',
    'Low Protein / Low Sugar':              '#95a5a6'
}

fig = px.scatter(
    sample,
    x='sugars_100g', y='proteins_100g',
    color='quadrant',
    color_discrete_map=quadrant_colors,
    facet_col='primary_category',
    facet_col_wrap=4,
    opacity=0.5,
    title='🧪 Nutrient Matrix: Sugar vs Protein by Category',
    labels={
        'sugars_100g': 'Sugar (g per 100g)',
        'proteins_100g': 'Protein (g per 100g)'
    },
    hover_data=['product_name', 'brands']
)

# Quadrant lines
fig.add_vline(x=sugar_median,   line_dash='dash', line_color='white', opacity=0.5)
fig.add_hline(y=protein_median, line_dash='dash', line_color='white', opacity=0.5)

fig.update_layout(height=700, paper_bgcolor='rgba(0,0,0,0)')
fig.show()
fig.write_html('story3_nutrient_matrix.html')
print('✅ Saved story3_nutrient_matrix.html')

✅ Saved story3_nutrient_matrix.html


In [52]:
# ─────────────────────────────────────────────────────────────
# Story 3: By-Category Quadrant Distribution
# ─────────────────────────────────────────────────────────────
quadrant_by_cat = (
    df.groupby(['primary_category', 'quadrant'])
    .size()
    .reset_index(name='count')
)
total_by_cat = df.groupby('primary_category').size().reset_index(name='total')
quadrant_by_cat = quadrant_by_cat.merge(total_by_cat, on='primary_category')
quadrant_by_cat['pct'] = (quadrant_by_cat['count'] / quadrant_by_cat['total'] * 100).round(1)

fig2 = px.bar(
    quadrant_by_cat,
    x='primary_category', y='pct',
    color='quadrant',
    color_discrete_map=quadrant_colors,
    title='📊 Quadrant Distribution by Category (%)',
    labels={'pct': '% Products', 'primary_category': 'Category'},
    text='pct',
    barmode='stack'
)
fig2.update_traces(texttemplate='%{text:.0f}%', textposition='inside')
fig2.update_layout(height=500, xaxis_tickangle=-30)
fig2.show()
fig2.write_html('story3_quadrant_by_category.html')
print('✅ Saved')

✅ Saved


---
## Story 4: The Market Gap Recommendation 🎯
**Goal:** Identify and quantify the Blue Ocean opportunity.

In [53]:
# ─────────────────────────────────────────────────────────────
# Step 4: Find the "Empty Quadrant" — High Protein, Low Sugar
# ─────────────────────────────────────────────────────────────
target_products = df[df['quadrant'] == '🌟 High Protein / Low Sugar (TARGET)'].copy()
crowded_products = df[df['quadrant'] == 'Low Protein / High Sugar (CROWDED)'].copy()

print('=== Market Gap Analysis ===')
print(f'Total snack products analyzed:    {len(df):,}')
print(f'High Protein / Low Sugar (Gap):   {len(target_products):,} ({len(target_products)/len(df)*100:.1f}%)')
print(f'Low Protein / High Sugar (Crowded): {len(crowded_products):,} ({len(crowded_products)/len(df)*100:.1f}%)')

print('\n=== Target Zone Nutritional Profile ===')
display(target_products[['sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g']].describe().round(1))

# Best category for opportunity
gap_by_cat = (
    df.groupby('primary_category')
    .apply(lambda x: (x['quadrant'] == '🌟 High Protein / Low Sugar (TARGET)').mean() * 100)
    .reset_index(name='target_pct')
    .sort_values('target_pct', ascending=False)
)
print('\n=== Target Zone % by Category ===')
display(gap_by_cat)

=== Market Gap Analysis ===
Total snack products analyzed:    30,189
High Protein / Low Sugar (Gap):   9,584 (31.7%)
Low Protein / High Sugar (Crowded): 9,545 (31.6%)

=== Target Zone Nutritional Profile ===


,sugars_100g,proteins_100g,fat_100g,fiber_100g
count,9584.00,9584.00,9574.00,8620.00
mean,3.40,11.70,11.60,5.00
std,2.40,5.30,16.10,5.40
min,0.00,7.00,0.00,0.00
25%,1.70,8.50,2.00,2.00
50%,3.50,10.00,3.80,3.60
75%,5.30,12.50,14.30,6.70
max,7.70,47.10,88.20,53.60



=== Target Zone % by Category ===


,primary_category,target_pct
6,Nuts & Seeds,53.23
1,Cereals & Granola,52.45
2,Chips & Savoury Snacks,38.90
4,Dairy & Yogurt,13.73
7,Other Snacks,8.18
8,Protein & Health Bars,5.58
0,Biscuits & Cookies,5.25
3,Chocolate & Candy,3.30
5,Dried Fruit & Fruit Snacks,1.51


In [54]:
# ─────────────────────────────────────────────────────────────
# The Recommendation
# ─────────────────────────────────────────────────────────────
best_cat  = gap_by_cat.iloc[0]['primary_category']
gap_pct   = gap_by_cat.iloc[0]['target_pct']

# Target nutritional benchmarks
target_protein = target_products['proteins_100g'].quantile(0.75).round(1)
target_sugar   = target_products['sugars_100g'].quantile(0.25).round(1)

print('=' * 65)
print('              🎯 MARKET RECOMMENDATION FOR HELIX CPG')
print('=' * 65)
print()
print('KEY INSIGHT:')
print(f'"Based on the data, the biggest market opportunity is in')
print(f' {best_cat}, specifically targeting products with')
print(f' {target_protein}g of protein and less than {target_sugar}g of sugar."')
print()
print(f'  • Only {gap_pct:.1f}% of products in this category occupy the')
print(f'    High Protein / Low Sugar quadrant — meaning the market')
print(f'    is severely under-served in this space.')
print()
print(f'  • The "Low Protein / High Sugar" quadrant is over-crowded:')
print(f'    {len(crowded_products)/len(df)*100:.1f}% of all products fall here.')
print('=' * 65)

              🎯 MARKET RECOMMENDATION FOR HELIX CPG

KEY INSIGHT:
"Based on the data, the biggest market opportunity is in
 Nuts & Seeds, specifically targeting products with
 12.5g of protein and less than 1.7g of sugar."

  • Only 53.2% of products in this category occupy the
    High Protein / Low Sugar quadrant — meaning the market
    is severely under-served in this space.

  • The "Low Protein / High Sugar" quadrant is over-crowded:
    31.6% of all products fall here.


In [55]:
# ─────────────────────────────────────────────────────────────
# Story 4 Visualization: Market Opportunity Bubble Chart
# ─────────────────────────────────────────────────────────────
opp_data = df.groupby('primary_category').agg(
    avg_sugar   = ('sugars_100g', 'mean'),
    avg_protein = ('proteins_100g', 'mean'),
    product_count = ('product_name', 'count'),
    target_pct  = ('quadrant', lambda x: (x == '🌟 High Protein / Low Sugar (TARGET)').mean() * 100)
).reset_index()

fig = px.scatter(
    opp_data,
    x='avg_sugar', y='avg_protein',
    size='product_count',
    color='target_pct',
    color_continuous_scale='RdYlGn',
    text='primary_category',
    title='🫧 Market Opportunity Map — Category Positioning',
    labels={
        'avg_sugar': 'Avg Sugar (g/100g)',
        'avg_protein': 'Avg Protein (g/100g)',
        'target_pct': '% in Target Zone'
    },
    size_max=60
)
fig.add_vline(x=sugar_median,   line_dash='dash', annotation_text='Sugar Median')
fig.add_hline(y=protein_median, line_dash='dash', annotation_text='Protein Median')

# Shade the target quadrant
fig.add_shape(type='rect',
    x0=0, y0=protein_median, x1=sugar_median, y1=100,
    fillcolor='rgba(46,204,113,0.1)', line_width=0
)
fig.add_annotation(
    x=sugar_median/2, y=protein_median + 5,
    text='🌟 BLUE OCEAN<br>OPPORTUNITY',
    showarrow=False, font=dict(size=11, color='#2ecc71')
)

fig.update_traces(textposition='top center')
fig.update_layout(height=550)
fig.show()
fig.write_html('story4_opportunity_map.html')
print('✅ Saved story4_opportunity_map.html')

✅ Saved story4_opportunity_map.html


---
## Bonus: The "Hidden Gem" — Protein Source Ingredients 🔬
**Goal:** Identify the top 3 protein sources in High Protein / Low Sugar products.

In [56]:
# ─────────────────────────────────────────────────────────────
# Extract top protein sources from ingredients_text
# ─────────────────────────────────────────────────────────────
PROTEIN_SOURCES = [
    'whey', 'peanut', 'pea protein', 'soy protein', 'soy',
    'almond', 'cashew', 'chickpea', 'lentil', 'quinoa',
    'egg', 'milk protein', 'casein', 'hemp', 'chia',
    'sunflower seed', 'pumpkin seed', 'sesame', 'flaxseed',
    'brown rice protein', 'oat', 'walnut', 'pistachio'
]

target_with_ingredients = target_products[target_products['ingredients_text'].notna()].copy()
target_with_ingredients['ingredients_lower'] = target_with_ingredients['ingredients_text'].str.lower()

# Count appearances of each protein source
source_counts = {}
for source in PROTEIN_SOURCES:
    count = target_with_ingredients['ingredients_lower'].str.contains(source, regex=False).sum()
    source_counts[source] = count

protein_df = pd.DataFrame(
    sorted(source_counts.items(), key=lambda x: -x[1]),
    columns=['Protein Source', 'Product Count']
)

print('=== Top Protein Sources in High Protein / Low Sugar Products ===')
display(protein_df.head(10))

top3 = protein_df.head(3)['Protein Source'].tolist()
print(f'\n🏆 Top 3 Protein Sources: {top3[0].title()}, {top3[1].title()}, {top3[2].title()}')

=== Top Protein Sources in High Protein / Low Sugar Products ===


,Protein Source,Product Count
0,soy,3742
1,oat,1004
2,sesame,763
3,whey,679
4,sunflower seed,362
5,peanut,333
6,egg,263
7,flaxseed,217
8,almond,204
9,quinoa,178



🏆 Top 3 Protein Sources: Soy, Oat, Sesame


In [57]:
# ─────────────────────────────────────────────────────────────
# Bonus Chart: Protein Sources
# ─────────────────────────────────────────────────────────────
fig = px.bar(
    protein_df.head(10),
    x='Product Count', y='Protein Source',
    orientation='h',
    color='Product Count',
    color_continuous_scale='Greens',
    title='🥦 Top Protein Sources in High-Protein / Low-Sugar Products',
    text='Product Count'
)
fig.update_traces(textposition='outside')
fig.update_layout(height=450, yaxis={'categoryorder':'total ascending'}, coloraxis_showscale=False)
fig.show()
fig.write_html('bonus_protein_sources.html')
print('✅ Saved')

✅ Saved


---
## Candidate's Choice: Nutri-Score vs Market Positioning 🏅

**What it is:** Cross-reference official Nutri-Score grades (A–E) with each product's Sugar/Protein quadrant.

**Why it matters to the business:**  
Regulators in France, Germany, and Belgium now mandate Nutri-Score labels on food products, and more countries are adopting it. A product targeting the High Protein / Low Sugar quadrant must *also* achieve a Nutri-Score of A or B to succeed on retail shelves in these markets. This analysis tells the R&D team what nutritional benchmarks are needed for *both* commercial and regulatory success.

In [58]:
# ─────────────────────────────────────────────────────────────
# Nutri-Score vs Quadrant analysis
# ─────────────────────────────────────────────────────────────
nutri_df = df[df['nutrition_grade_fr'].notna()].copy()
nutri_df['nutrition_grade_fr'] = nutri_df['nutrition_grade_fr'].str.upper().str.strip()
nutri_df = nutri_df[nutri_df['nutrition_grade_fr'].isin(['A', 'B', 'C', 'D', 'E'])]

# Cross-tabulation: Nutri-Score Grade vs Quadrant
cross = pd.crosstab(
    nutri_df['nutrition_grade_fr'],
    nutri_df['quadrant'],
    normalize='index'
) * 100

cross = cross.round(1)
print('=== % of products in each quadrant, by Nutri-Score grade ===')
display(cross)

# Plot
cross_long = cross.reset_index().melt(id_vars='nutrition_grade_fr', var_name='Quadrant', value_name='Pct')

fig = px.bar(
    cross_long,
    x='nutrition_grade_fr', y='Pct',
    color='Quadrant',
    color_discrete_map=quadrant_colors,
    title='🏅 Nutri-Score Grade vs Nutrient Quadrant',
    labels={'nutrition_grade_fr': 'Nutri-Score Grade', 'Pct': '% of Products'},
    barmode='stack',
    text='Pct'
)
fig.update_traces(texttemplate='%{text:.0f}%', textposition='inside')
fig.update_layout(height=500)
fig.show()
fig.write_html('choice_nutriscore.html')
print('✅ Saved choice_nutriscore.html')

KeyError: 'nutrition_grade_fr'

In [ ]:
# ─────────────────────────────────────────────────────────────
# Export clean data for dashboard
# ─────────────────────────────────────────────────────────────
export_cols = [
    'product_name', 'brands', 'primary_category', 'quadrant',
    'sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g',
    'nutrition_grade_fr', 'ingredients_text'
]

clean_export = df[export_cols].copy()
clean_export.to_csv('openfoodfacts_clean.csv', index=False)

opp_data.to_csv('opportunity_map.csv', index=False)
protein_df.to_csv('protein_sources.csv', index=False)

print(f'✅ Exported openfoodfacts_clean.csv — {len(clean_export):,} products')
print('✅ Exported opportunity_map.csv')
print('✅ Exported protein_sources.csv')

print(f'\n=== FINAL SUMMARY ===')
print(f'Products analyzed:          {len(df):,}')
print(f'Categories identified:      {df["primary_category"].nunique()}')
print(f'High Protein / Low Sugar:   {len(target_products):,} ({len(target_products)/len(df)*100:.1f}%)')
print(f'Low Protein / High Sugar:   {len(crowded_products):,} ({len(crowded_products)/len(df)*100:.1f}%)')
print(f'\nTop 3 Protein Sources: {top3}')
print(f'\nRECOMMENDATION: Launch in [{best_cat}]')
print(f'Target: ≥{target_protein}g protein, ≤{target_sugar}g sugar per 100g')

In [ ]:
from google.colab import files

# Export all 3 CSVs
df[['product_name', 'brands', 'primary_category', 'quadrant',
    'sugars_100g', 'proteins_100g', 'fat_100g', 'fiber_100g',
    'nutrition_grade_fr', 'ingredients_text']].to_csv('openfoodfacts_clean.csv', index=False)

opp_data.to_csv('opportunity_map.csv', index=False)
protein_df.to_csv('protein_sources.csv', index=False)

# Download them directly to your computer
files.download('openfoodfacts_clean.csv')
files.download('opportunity_map.csv')
files.download('protein_sources.csv')

print('✅ All 3 CSVs downloaded!')